In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("titanic_cleaned_dataset.csv")
df.head()


,Unnamed: 0,PassengerId,Survived,Pclass,Name,Sex,Age,Siblings,Parents,Fare,Embarked
0,0,1,0,3,"Braund, Mr. Owen Harris",male,22,1,0,7.2500,S
1,2,3,1,3,"Heikkinen, Miss. Laina",female,26,0,0,7.9250,S
2,3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35,1,0,53.1000,S
3,4,5,0,3,"Allen, Mr. William Henry",male,35,0,0,8.0500,S
4,5,6,0,3,"Moran, Mr. James",male,29,0,0,8.4583,Q


In [2]:
#We generate a data quality report to identify missing values, duplicates, and dtype issues.

report = pd.DataFrame({
    "Nulls": df.isnull().sum(),
    "Dtype": df.dtypes,
    "Unique Values": df.nunique()
})
print("Duplicate rows:", df.duplicated().sum())
report


Duplicate rows: 0


,Nulls,Dtype,Unique Values
Unnamed: 0,0,int64,775
PassengerId,0,int64,775
Survived,0,int64,2
Pclass,0,int64,3
Name,0,str,775
Sex,0,str,2
Age,0,int64,71
Siblings,0,int64,6
Parents,0,int64,7
Fare,0,float64,203


In [7]:

#Median imputation for Age (robust to skew), mode for Embarked (categorical), and Cabin dropped due to excessive missingness.

df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

In [6]:
# removed duplicate rows to ensure data integrity.
before = len(df)
df.drop_duplicates(inplace=True)
after = len(df)
print("Removed duplicates:", before - after)



Removed duplicates: 0


In [8]:
# Normalize Sex
df['Sex'] = df['Sex'].str.strip().str.lower().replace({
    'male':'Male','m':'Male','female':'Female','f':'Female'
})
# Normalize Embarked
df['Embarked'] = df['Embarked'].str.upper().replace({
    'C':'C','Q':'Q','S':'S'
})


In [9]:
Q1 = df['Fare'].quantile(0.25)
Q3 = df['Fare'].quantile(0.75)
IQR = Q3 - Q1

outliers = df[(df['Fare'] < (Q1 - 1.5*IQR)) | (df['Fare'] > (Q3 + 1.5*IQR))]
print("Outliers detected:", len(outliers))

# Cap extreme fares
df['Fare'] = np.where(df['Fare'] > (Q3 + 1.5*IQR), Q3 + 1.5*IQR, df['Fare'])


Outliers detected: 25


In [10]:
df['PassengerId'] = df['PassengerId'].astype(str)
df['Survived'] = df['Survived'].astype('category')
df['Fare'] = df['Fare'].astype(float)
